# Building a Decision Tree

In this section, we explore how to build Decision Tree step by step when we have a large dataset. And we will focus on how different splitting criteria is calculated

At each node in the tree, the algorithm performs three main operations:
1. **Finding the best split:**

    The algorithm searches through all features and possible threshold values to determine the split that best separates the data.
2. **Split the dataset:**

    Once the optimal feature and threshold are selected, the data is divided into two groups:

    - The left side of the branch: data that satisfies the condition in the branch
    - The right side of the branch: data that do **NOT** satisfy the condition in the branch

3. **Repeat recursively:**

    The 2 above processes are repeated, creating new nodes and further splits.

    It continues until a stopping condition is met, such as:
    - Pure node: all data in that node belongs to the same class
    - The tree depth has reached its maximum
    - Or there isn’t enough data left in that node to make another useful split.

---

## Exploring the Decision Tree

### A simple example:
Let's say you're trying to predict whether a person's loan will get approved at the bank. You were able to collect the data below from previous customers who applied for loans, along with the outcome of each application (Approved or Rejected).

There are 2 features within your dataset:
- Feature 1: How much debt a person had when they applied for the loan
- Feature 2: How much they were spending at the time of applying

The tree starts by calculating which feature and split will best separate the data into two groups. In simple terms, it checks: “If I split the data here, does it separate approvals and rejections well?”

It tries different thresholds for each feature (like different debt levels or spending amounts) and measures which split gives the cleanest separation between the two classes.


You can use the interactive widget below to see a Decision Tree in action

*Suggestion use:*
- *Press the "Auto-grow" button to see how the tree splits the data, and grow new nodes*
- *Hover over each node from top to bottom to see which splits result in which nodes*
- *Try different tree depths to see how it affects the tree structure and Training/Testing accuracy*
- *Add more data using the buttons next to "Feature space" to see how it affects the growth of the tree*

In [ ]:
from IPython.display import HTML, display

with open("decision_tree.html", "r", encoding="utf-8") as f:
    html_content = f.read()

wrapped_html = f"""
<div style="width: 100%; overflow-x: auto;">
    {html_content}
</div>
"""

display(HTML(wrapped_html))

### A note on numerical data

**2D data:**
In this example, each dataset has only 2 features. This is called 2-dimensional (2D) data.

We can think of each data point as a position on a flat plane, like a map:
- One axis represents debt
- The other axis represents spending

Because of this, we can actually visualize the decision tree splits. Each split becomes a straight line:
- A vertical line if the split is based on the feature in the Y-axis (Debt)
- A horizontal line if the split is based on the other feature in the X-axis (Spending)

This makes it easier to see how the algorithm divides the space into regions that correspond to different predictions.

**3D data:** As for data with 3 features, instead of splitting with lines, a decision tree now splits using flat surfaces (planes) that cut through this space.

**Higher-dimensional data:** For data with 4 or more features, we can no longer visualize it. But the key idea remains the same, the tree keeps finding the best feature and threshold to make splits

### What about categorical data?
Some features are not numerical, but instead have categorical values. For these features, the split is very simple: you'll only need one branch for each category.
But the goal remains the same: create groups that are as **pure** as possible.

---

## How to pick the best split?
This is one of the most important parts of a decision tree, because every split directly affects how well the model learns patterns in the data.

To decide which split is best, we need a way to measure how good a separation is. In general, we want each resulting group to be as pure as possible, meaning most data points in a group belong to the same class. This is done by measuring the impurity (or disorder) of a node.

There are three common ways to measure this impurity:
- Gini impurity
- Entropy & Information gain
- Misclassification cost

Each one gives a slightly different way of measuring how diverse the classes are in a node. The Decision Tree uses one of these methods to evaluate all possible splits and then chooses the split that produces the cleanest (most pure) separation.

## What is Gini impurity?
Suppose you have a box of different colored shapes. Each element represents a data point, and the color represents its class (label).
This box is like a node in a decision tree, which contains a group of data points.

Now imagine you randomly pick an element from the box, put it back (so overlapping is allowed), and then pick one more element.

Gini impurity measures the probability that the **two elements you picked have distinct colors**, in other words they are from different classes.

Example: Inside your box, there are:

- 4 blue squares
- 3 red circles
- 2 green triangles
- 1 yellow star

Total = 10 elements

The Gini impurity is computed using the formula shown in the figure below. Source: @Luis_Serrano_Academy_2021

![alt text](figures/gini_index.png)

**General formula:** $$Gini = 1 - \sum_i p_i^2$$

where $p_i$ is the **probability of class i** in the node

```{tip} So what is the Gini impurity in our example?
:class: dropdown
:open: false
:icon: true

$Gini = 1 - \sum p_i^2 = 1 - \left(\frac{4}{10}\right)^2 - \left(\frac{3}{10}\right)^2 - \left(\frac{2}{10}\right)^2 - \left(\frac{1}{10}\right)^2 = \frac{7}{10}$
```

## What is Entropy & Information gain?
In the simplest terms, Entropy measures how much we can “rearrange” or vary a set to get different sequences.
Suppose we have 3 buckets of 4 balls each, with the following colors:

![alt text](figures/buckets.png)

As you can probably guess, there are very few possible distinct sequences you can form from bucket 1, some from bucket 2, and many more from bucket 3.

Therefore, we say:

- Bucket 1 has low entropy
- Bucket 2 has medium entropy
- Bucket 3 has high entropy

Again, let's imagine you randomly pick a ball from each bucket, put it back, and repeat this process 4 times. What is the probability of generating the exact same sequence of colors that is already shown in the box? Because each time, you randomly pick a ball from the bucket is an independent event, the probability is computed as below:

![alt text](figures/probability.png)

However, looking at the probability, there are 2 slight problems:
- The calculated probability of each bucket behaves in the opposite way to its entropy: bucket with low entropy has high probability of generating the exact sequence
- The more data we have, the smaller this probability will become

This is where logarithms become useful. Not only do they turn products into sums, but they also make very small probabilities easier to work with.

Because probabilities of long sequences become extremely small (since we multiply many numbers between 0 and 1), taking the logarithm transforms them into a more manageable scale and allows us to compare them more easily.

Logarithm identity:
$$\log_2(xy) = \log_2(x) + \log_2(y)$$

And if we take the logarithm of the probability (and multiply by -1 to make the values positive), we can rewrite the expression in a more useful form. Now, as a final step, we take the average to normalize the value.

![alt text](figures/entropy.png)

**General formula:** $$Entropy = -\sum_i p_i \log(p_i)$$

```{tip} So what is the Entropy in our example?
:class: dropdown
:open: false
:icon: true


Bucket 1:
$$Entropy = -\sum_i p_i \log(p_i) = \frac{1}{4} \left( -\log_2(1) - \log_2(1) - \log_2(1) - \log_2(1) \right) = 0$$

Bucket 2:
$$Entropy = -\sum_i p_i \log(p_i) = \frac{1}{4} \left( -\log_2(0.75) - \log_2(0.75) - \log_2(0.75) - \log_2(0.25) \right) = 0.81125$$

Bucket 3:
$$Entropy = -\sum_i p_i \log(p_i) =  \frac{1}{4} \left( -\log_2(0.5) - \log_2(0.5) - \log_2(0.5) - \log_2(0.5) \right) = 1$$
```

While entropy measures how diverse (uncertain) or mixed a node is, it is not used directly as the final splitting criterion in a decision tree. Instead, decision trees use information gain, which measures how much that uncertainty is reduced after a split.

To compute **Information gain**, we compare the entropy of the data before the split with the Entropy **after the split**.

### Step 1: Entropy before the split

We compute the entropy of the original node:

$$
H(\text{parent}) = - \sum_i p_i \log_2(p_i)
$$

This measures how mixed the data is before any split happens.

### Step 2: Entropy after the split

After splitting the data into child nodes (e.g., left and right), we compute the entropy of each child:

$$
H(L), \quad H(R)
$$

Since child nodes can have different sizes, we take a weighted average:

$$
H(\text{after}) = \frac{N_L}{N} H(L) + \frac{N_R}{N} H(R)
$$

where:
- $N_L, N_R$ are the number of samples in each child node  
- $N$ is the total number of samples  

### Step 3: Information gain

Finally, we compute Information gain ($IG$):

$$
IG = H(\text{parent}) - H(\text{after})
$$


## What is misclassification cost?
Another way to measure how good a split is comes from a very simple idea: "What happens if we just make a wrong guess?"

For a given node, we first pick the most common class, and then assume we always predict that class. The misclassification cost is simply the fraction of data points that would be incorrectly labeled.

So:

If almost everything belongs to one class -> low cost
If classes are very mixed -> high cost

**General formula:**
$$Misclassification = 1 - \max_i p_i$$

## Summary
In this chapter, we explored how decision trees learn by repeatedly splitting data into smaller and purer groups. At each step, the model tests many possible splits and uses a scoring rule to decide which one is best.

We saw three common splitting criteria:
- Gini impurity, which measures how mixed the classes are
- Entropy (and information gain), which measures how much uncertainty is reduced after a split
- Misclassification cost, which measures how many simple prediction errors would occur

Even though these methods look different, they are all used in the same way: the tree evaluates each possible split, computes the chosen score for the resulting groups, and selects the split that produces the “cleanest” separation of the data.

Overall, decision trees build structure step by step by always choosing the split that best organizes the data at each node.